## 1. Persistence Baseline

In [18]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [14]:
import json

from src.dataloader import load_arrays, load_cgm_stats
from src.metrics import evaluate, format_results_table
from src.models.classical_baseline import PersistencePredictor

MODEL_NAME = 'Persistence'
# Built from REPO_ROOT, not the current working directory: Colab's cwd is /content regardless of
# where this notebook file itself lives on the mounted Drive, so a plain relative path like
# Path('results') would silently write to the wrong (ephemeral) place.
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_PATH = EXPERIMENT_DIR / 'results' / 'persistence_results.json'

In [15]:
stats = load_cgm_stats()
X_train, y_train = load_arrays('train')
X_val, y_val = load_arrays('val')
X_test, y_test = load_arrays('test')

print('train', X_train.shape, 'val', X_val.shape, 'test', X_test.shape)

train (83532, 24) val (16739, 24) test (16842, 24)


In [16]:
model = PersistencePredictor().fit(X_train, y_train)  # nothing to fit -- zero parameters

In [17]:
val_results = evaluate(model.predict(X_val), y_val, stats['cgm_mean'], stats['cgm_std'])
test_results = evaluate(model.predict(X_test), y_test, stats['cgm_mean'], stats['cgm_std'])

print('=== Validation ===')
print(format_results_table(val_results))
print()
print('=== Test ===')
print(format_results_table(test_results))

=== Validation ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    12.31 |     8.15 |    13.31 |      99.79
           30 |    19.40 |    12.74 |    21.40 |      99.43
           60 |    27.95 |    18.27 |    31.58 |      98.60
           90 |    33.72 |    22.16 |    38.34 |      97.74
          120 |    37.80 |    25.30 |    42.99 |      97.08

=== Test ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    12.78 |     8.33 |    14.10 |      99.86
           30 |    20.49 |    13.23 |    23.20 |      99.65
           60 |    30.43 |    19.47 |    35.32 |      98.98
           90 |    37.39 |    24.03 |    43.73 |      98.13
          120 |    42.09 |    27.34 |    49.28 |      97.22


In [12]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seed': None,  # deterministic, no seed applies
        'val_results': val_results,
        'test_results': test_results,
    }, f, indent=2)

print(f'saved to {RESULTS_PATH}')

saved to /Users/ucabtt4/Projects/Dissertation/dissertation/experiments/1_baseline/results/persistence_results.json
